In [1]:
import json
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from massive import RESTClient

In [ ]:
with open("api_key.json") as f:
    api_keys = json.load(f)

client = RESTClient(api_keys["massive_api_key"])
del api_keys

In [3]:
from datetime import datetime, timedelta

# Calculate date range for past 1.5 years
end_date = datetime.now()
start_date = end_date - timedelta(days=int(1.5 * 365))

aggs = []
for a in client.list_aggs(
    "NVDA",
    1,
    "day",
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d"),
    limit=50000,
):
    aggs.append(a)

print(f"Downloaded {len(aggs)} data points")
print(f"Date range: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")

Downloaded 376 data points
Date range: 2024-06-04 to 2025-12-03


In [4]:
# Convert to DataFrame
df = pd.DataFrame(aggs)
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')

df.head()

,open,high,low,close,volume,vwap,timestamp,transactions,otc
0,115.716,116.6000,114.045,116.437,403324010.0,115.2975,2024-06-04 04:00:00,1056061,None
1,118.371,122.4495,117.468,122.440,528401780.0,120.4210,2024-06-05 04:00:00,1410118,None
2,124.048,125.5870,118.320,120.998,664696190.0,121.1279,2024-06-06 04:00:00,1912084,None
3,119.770,121.6917,118.022,120.888,412385800.0,120.0427,2024-06-07 04:00:00,1119491,None
4,120.370,195.9500,117.010,121.790,314162650.0,121.1155,2024-06-10 04:00:00,2798759,None


In [5]:
# Create interactive plot with plotly
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=('NVDA Price', 'Volume'),
    row_heights=[0.7, 0.3]
)

# Add candlestick chart
fig.add_trace(
    go.Candlestick(
        x=df['timestamp'],
        open=df['open'],
        high=df['high'],
        low=df['low'],
        close=df['close'],
        name='NVDA'
    ),
    row=1, col=1
)

# Add volume bar chart
fig.add_trace(
    go.Bar(
        x=df['timestamp'],
        y=df['volume'],
        name='Volume',
        marker_color='rgba(0, 150, 255, 0.5)'
    ),
    row=2, col=1
)

# Update layout
fig.update_layout(
    title='NVDA Stock Price and Volume',
    yaxis_title='Price (USD)',
    yaxis2_title='Volume',
    xaxis2_title='Date',
    height=800,
    xaxis_rangeslider_visible=False
)

# Export to HTML
fig.write_html('nvda_chart.html')
print('Chart exported to nvda_chart.html')

fig.show()

Chart exported to nvda_chart.html
